# 3. Análisis Exploratorio de Datos (EDA)
## TFM — Predicción del riesgo de readmisión hospitalaria en pacientes diabéticos

Dataset: *Diabetes 130-US Hospitals for Years 1999-2008* (UCI Machine Learning Repository).

En este notebook iremos construyendo el EDA paso a paso, sección por sección, siguiendo el índice de la propuesta:

1. Carga de datos y primer vistazo
2. Análisis descriptivo univariante y bivariante
3. Distribución de la variable objetivo (desbalanceo de clases)
4. Relación entre variables clínicas/demográficas y readmisión
5. Detección de valores ausentes, outliers e inconsistencias

## 3.1 Carga de librerías y del dataset

Empezamos importando las librerías que vamos a necesitar para esta primera fase (manejo de datos y visualización básica) y cargando el CSV en un DataFrame de pandas.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 60)
sns.set_style('whitegrid')

In [4]:
# pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

# id=296 -> Diabetes 130-US Hospitals for Years 1999-2008
dataset = fetch_ucirepo(id=296)

X = dataset.data.features      # variables predictoras
y = dataset.data.targets       # variable "readmitted"

# metadatos y cita
print(dataset.metadata.additional_info.summary)
print(dataset.variables)

The dataset represents ten years (1999-2008) of clinical care at 130 US hospitals and integrated delivery networks. It includes over 50 features representing patient and hospital outcomes. Information was extracted from the database for encounters that satisfied the following criteria.
(1)	It is an inpatient encounter (a hospital admission).
(2)	It is a diabetic encounter, that is, one during which any kind of diabetes was entered into the system as a diagnosis.
(3)	The length of stay was at least 1 day and at most 14 days.
(4)	Laboratory tests were performed during the encounter.
(5)	Medications were administered during the encounter.

The data contains such attributes as patient number, race, gender, age, admission type, time in hospital, medical specialty of admitting physician, number of lab tests performed, HbA1c test result, diagnosis, number of medications, diabetic medications, number of outpatient, inpatient, and emergency visits in the year before the hospitalization, etc.
  

C:\Users\lballi01\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [9]:
# Unimos en un único DataFrame para trabajar cómodamente
df = pd.concat([dataset.data.ids, X, y], axis=1)

print(f"Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")

Filas: 101,766  |  Columnas: 50


In [10]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      99493 non-null   object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    3197 non-null    object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                61510 non-null   object
 11  medical_specialty         51817 non-null   object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

## Conclusiones del primer análisis exploratorio (carga y estructura del dataset)

**1. Estructura general**

- El dataset contiene **101.766 filas** (encuentros hospitalarios) y **50 columnas**.
- 13 variables numéricas (`int64`) y 37 categóricas/texto (`object`). Muchas de estas últimas son en realidad categóricas de pocos niveles (medicamentos con valores `No/Steady/Up/Down`, género, etc.), no texto libre.

**2. Valores ausentes**

Gracias al uso de `ucimlrepo`, los valores ausentes ya vienen codificados como `NaN` (en el CSV original de UCI se representan con el carácter `"?"`).

| Columna | % ausente | Comentario |
|---|---|---|
| `weight` | 96.9% | Prácticamente inutilizable tal cual; candidata a eliminar o convertir en indicador binario ("¿se registró peso?"). |
| `max_glu_serum` | 94.8% | Solo se realizó esta prueba a ~5% de los pacientes. |
| `A1Cresult` | 83.3% | Prueba minoritaria, pero clínicamente relevante para diabetes. |
| `medical_specialty` | 49.1% | Casi la mitad de los registros sin especialidad médica asociada. |
| `payer_code` | 39.6% | Ausencia considerable; poca relevancia clínica esperada. |
| `race` | 2.2% | Ausencia baja, fácilmente tratable (imputación o eliminación de filas). |
| `diag_3` | 1.4% | Ausencia baja. |
| `diag_2` | 0.35% | Ausencia muy baja. |
| `diag_1` | 0.02% | Prácticamente completa. |

El resto de columnas (variables numéricas, fármacos, `readmitted`, etc.) no presentan valores ausentes.

**3. Primeras decisiones que se derivan de este análisis** *(a formalizar en la sección de Preparación y transformación de datos)*

- `weight`: candidata a eliminar o transformar en variable binaria, dado su altísimo porcentaje de ausencia (97%).
- `max_glu_serum` y `A1Cresult`: pese a su alta tasa de ausencia, no deben descartarse sin más — son clínicamente relevantes en diabetes, y el hecho de que "no se haya realizado la prueba" puede ser en sí mismo informativo. Se tratarán probablemente mediante una categoría explícita tipo `"No testeado"`.
- `payer_code`: buena candidata a eliminación por su bajo valor predictivo esperado.
- `medical_specialty`: caso intermedio; se valorará agrupar en categorías amplias (p. ej. "Cardiología", "Medicina Familiar", "Desconocido") en lugar de descartarla directamente.

Esta cuantificación de valores ausentes constituye la base del apartado de "detección de valores ausentes, outliers e inconsistencias" del EDA.